# 04 -- MP vs Stage III/IV: Differential Abundance & Cross-Sectional Trajectory

**Purpose:** characterise the "MP" Study.Group -- **multiple polypoid adenomas** (Yachida
et al. 2019), a cancer-free precursor lesion -- contrasted against the most advanced
diagnosed disease stage (Stage III/IV):

1. Differential abundance, `Stage_III_IV` vs `MP` -- species and metabolites (volcano plots).
2. Polyamine abundance across the five `Study.Group` categories, ordered by
   `utils.CRC_STAGE_ORDER` (the corrected clinical severity ordering).
3. A small classifier distinguishing `MP` from `Stage_III_IV` samples, with
   cross-validated AUC.

**Dependency (honest statement):** this notebook is **not** standalone. It reads
`preprocessed_data_reduced_v2.pkl`, which is written by `02_analysis.ipynb`
(NB02 must be run first). The older version of this notebook claimed to be
independent of the rest of the pipeline; that was false -- it loaded exactly
this pickle.

**Framing correction:** the five `Study.Group` stage categories
(`Healthy, MP, Stage_0, Stage_I_II, Stage_III_IV`; HS -- history of colorectal surgery --
is excluded from v2) are **disjoint
cross-sectional patient groups**, not a tracked cohort followed over time --
every `Subject` in the raw Yachida metadata is unique. This notebook does not
describe or plot a "longitudinal trajectory" or "recovery" of individual
patients. It compares group-level distributions across an *ordinal severity
axis*, where `MP` (adenoma -- no cancer diagnosis, TNM stage `"-"`) sits adjacent to
`Healthy`, **not** past `Stage_III_IV`. See `utils.py`'s `STAGE ENCODING` block for the underlying
clinical justification.

**Downstream consumers:** none. This is a self-contained leaf analysis --
no other v2 notebook reads any file this notebook writes.


In [ ]:
import sys, pickle, warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
from sklearn.ensemble import RandomForestClassifier
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.model_selection import RepeatedStratifiedKFold, cross_val_score

warnings.filterwarnings('ignore')

sys.path.insert(0, '.')
import utils
from utils import (
    RESULTS_DIR, POLYAMINE_KEGG, CRC_STAGE_ORDER, CRC_STAGE_NUMERIC,
    differential_abundance, volcano_plot,
)

PREP_PKL = RESULTS_DIR / 'intermediate' / 'preprocessed_data_reduced_v2.pkl'
CRC_DS   = 'YACHIDA-CRC-2019'

TABLES_DIR = RESULTS_DIR / 'tables'
FIG_DIR    = RESULTS_DIR / 'figures' / 'differential'
TABLES_DIR.mkdir(parents=True, exist_ok=True)
FIG_DIR.mkdir(parents=True, exist_ok=True)

sns.set_style('whitegrid')
sns.set_context('paper', font_scale=1.2)

print("Imports OK")
print("Stage order (utils.CRC_STAGE_ORDER):", CRC_STAGE_ORDER)


In [ ]:
if not PREP_PKL.exists():
    raise FileNotFoundError(
        f"{PREP_PKL} not found -- run 02_analysis.ipynb first. "
        "This notebook depends on its reduced-feature output pickle."
    )

with open(PREP_PKL, 'rb') as f:
    _prep = pickle.load(f)

spc  = _prep['transformed_species'][CRC_DS].copy()
mtb  = _prep['transformed_mtb'][CRC_DS].copy()
meta = _prep['harmonized_meta'][CRC_DS].copy()
polyamine_cols = _prep['polyamine_columns'][CRC_DS]   # {KEGG_ID: col_name}

# Numeric coercion guard
feature_cols_spc = [c for c in spc.columns if c != 'Sample']
feature_cols_mtb = [c for c in mtb.columns if c != 'Sample']
for df, fcols in [(spc, feature_cols_spc), (mtb, feature_cols_mtb)]:
    df[fcols] = df[fcols].apply(pd.to_numeric, errors='coerce')

pa_cols = [c for c in polyamine_cols.values() if c in mtb.columns]
KEGG_TO_NAME = {v: POLYAMINE_KEGG[k] for k, v in polyamine_cols.items()}

# Sanity check: stage groups really are disjoint patients (cross-sectional),
# not repeated measures on the same subjects.
n_subjects = meta['Subject'].nunique() if 'Subject' in meta.columns else None
n_samples  = len(meta)
if n_subjects is not None:
    print(f"Samples: {n_samples}  |  Unique subjects: {n_subjects}  "
          f"({'cross-sectional, confirmed 1 sample/subject' if n_subjects == n_samples else 'NOTE: subjects repeat -- check before treating groups as independent'})")
else:
    print(f"Samples: {n_samples} (no Subject column available in this pickle's metadata)")

print(f"Stage_III_IV: {(meta['Study.Group'] == 'Stage_III_IV').sum()}")
print(f"MP:           {(meta['Study.Group'] == 'MP').sum()}")
print(f"Polyamine columns found ({len(pa_cols)}): {pa_cols}")


## 1. Differential abundance: Stage III/IV vs MP

`utils.differential_abundance(df, meta, group1, group2)` runs a Mann-Whitney U test per
feature with BH-FDR correction and reports **`MeanDiff` = mean(group2) − mean(group1)** in the
transformed space, plus Cliff's δ computed from U = #(group1 > group2) pairs, i.e.
**`CliffsD` > 0 means higher in group1**.

Here `group1='MP'`, `group2='Stage_III_IV'`, so:
- **`MeanDiff` > 0 = higher in Stage III/IV**
- `CliffsD` > 0 = higher in MP (opposite sign); the extra column
  `CliffsD_IIIIV_gt_MP` = −CliffsD has the same sign as `MeanDiff`.

(An earlier version called it with the groups the other way round while still labelling
positive `MeanDiff` as "higher in Stage III/IV" -- the opposite of what was computed.)

In [ ]:
# group1='MP', group2='Stage_III_IV'  ->  MeanDiff = mean(III/IV) - mean(MP) (> 0 = higher in III/IV)
mtb_da = differential_abundance(mtb, meta, 'MP', 'Stage_III_IV')
spc_da = differential_abundance(spc, meta, 'MP', 'Stage_III_IV')
for _df in (mtb_da, spc_da):
    if not _df.empty:
        _df['CliffsD_IIIIV_gt_MP'] = -_df['CliffsD']   # same sign as MeanDiff

pa_da = mtb_da[mtb_da['Feature'].isin(pa_cols)].copy()
pa_da['Metabolite_name'] = pa_da['Feature'].map(KEGG_TO_NAME)

print(f"=== Metabolite DA: Stage_III_IV vs MP  ({len(mtb_da)} features tested) ===")
print(f"Significant (q<0.05): {int(mtb_da['Significant'].sum()) if not mtb_da.empty else 0}")
if not pa_da.empty:
    print("\nPolyamine subset (MeanDiff, positive = higher in Stage III/IV):")
    print(pa_da[['Feature', 'Metabolite_name', 'MeanDiff', 'CliffsD_IIIIV_gt_MP', 'PValue', 'QValue', 'Significant']]
          .to_string(index=False))
else:
    print("\nNo polyamine columns present in the metabolite DA results.")

print(f"\n=== Species DA: Stage_III_IV vs MP  ({len(spc_da)} features tested) ===")
print(f"Significant (q<0.05): {int(spc_da['Significant'].sum()) if not spc_da.empty else 0}")


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 7))

if not mtb_da.empty:
    volcano_plot(mtb_da, 'Metabolites: Stage III/IV vs MP', fc_col='MeanDiff',
                 highlight_features=pa_cols if pa_cols else None, ax=axes[0])
else:
    axes[0].text(0.5, 0.5, 'No metabolite DA results', ha='center', transform=axes[0].transAxes)
    axes[0].set_title('Metabolites: Stage III/IV vs MP')

if not spc_da.empty:
    volcano_plot(spc_da, 'Species: Stage III/IV vs MP', fc_col='MeanDiff', ax=axes[1])
else:
    axes[1].text(0.5, 0.5, 'No species DA results', ha='center', transform=axes[1].transAxes)
    axes[1].set_title('Species: Stage III/IV vs MP')

plt.suptitle('Stage III/IV vs MP -- Differential Abundance (MeanDiff; positive = higher in Stage III/IV)',
             fontsize=13, fontweight='bold', y=1.02)
plt.tight_layout()

out_v = FIG_DIR / 'mp_vs_stageIIIIV_volcano_v2.png'
fig.savefig(out_v, dpi=150, bbox_inches='tight')
plt.show(); plt.close()
print(f"Saved: {out_v}")


## 2. Polyamine abundance across CRC stage groups (cross-sectional, ordinal)

Five **disjoint** patient groups, ordered along `utils.CRC_STAGE_ORDER` --
`Healthy, MP, Stage_0, Stage_I_II, Stage_III_IV`. `MP` (multiple polypoid adenomas) is a
cancer-free precursor (TNM stage `"-"`, same as `Healthy`) and sits adjacent to `Healthy`,
*before* any diagnosed cancer stage. HS (history of colorectal surgery) is excluded.

This is **not** a trajectory that any single patient travels along. It is a box-and-strip
plot of group-level distributions along an ordinal severity axis:
- A thin trend line is drawn **only** across the three diagnosed-cancer stages
  (`Stage_0 -> Stage_I_II -> Stage_III_IV`).
- `MP` is drawn with a distinct marker and labelled as an adenoma (precursor) group,
  separate from the diagnosed-cancer trend line.

In [ ]:
stage_order = CRC_STAGE_ORDER  # ['Healthy', 'MP', 'Stage_0', 'Stage_I_II', 'Stage_III_IV']
PRECURSOR_STAGES  = {'MP'}
DIAGNOSED_PROGRESSION = ['Stage_0', 'Stage_I_II', 'Stage_III_IV']  # monotonic trend only over these

stage_labels = {
    'Healthy': 'Healthy', 'MP': 'MP\n(adenoma)',
    'Stage_0': 'Stage 0', 'Stage_I_II': 'Stage I/II', 'Stage_III_IV': 'Stage III/IV',
}

mtb_meta = mtb.merge(meta[['Sample', 'Study.Group']], on='Sample')

n_pa = len(pa_cols)
if n_pa == 0:
    print("No polyamine columns available -- skipping trajectory figure (nothing to plot).")
else:
    ncols = min(3, n_pa)
    nrows = (n_pa + ncols - 1) // ncols  # safe: ncols >= 1 since n_pa >= 1 here

    fig, axes = plt.subplots(nrows, ncols, figsize=(5.5 * ncols, 4.2 * nrows), sharey=False, squeeze=False)
    axes_flat = axes.flatten()

    for i, col in enumerate(pa_cols):
        ax = axes_flat[i]
        pa_name = KEGG_TO_NAME.get(col, col)

        plot_df = mtb_meta[['Study.Group', col]].dropna(subset=[col])
        plot_df = plot_df[plot_df['Study.Group'].isin(stage_order)].copy()
        plot_df['Study.Group'] = pd.Categorical(plot_df['Study.Group'], categories=stage_order, ordered=True)

        box_colors = ['#a0a0a0' if sg == 'Healthy'
                      else '#e8b84b' if sg in PRECURSOR_STAGES
                      else '#4c72b0' for sg in stage_order]

        sns.boxplot(data=plot_df, x='Study.Group', y=col, order=stage_order,
                    palette=box_colors, ax=ax, linewidth=0.8, fliersize=2)
        sns.stripplot(data=plot_df, x='Study.Group', y=col, order=stage_order,
                      color='black', alpha=0.25, size=3, ax=ax)

        # Monotonic trend line drawn ONLY across the diagnosed-cancer stages --
        # explicitly not through MP, and not looping back from Stage_III_IV.
        medians_by_stage = plot_df.groupby('Study.Group', observed=True)[col].median()
        prog_x = [stage_order.index(s) for s in DIAGNOSED_PROGRESSION if s in medians_by_stage.index]
        prog_y = [medians_by_stage[s] for s in DIAGNOSED_PROGRESSION if s in medians_by_stage.index]
        if len(prog_x) >= 2:
            ax.plot(prog_x, prog_y, '-', color='darkred', lw=2, marker='D', ms=5,
                     zorder=4, label='Diagnosed-cancer trend')

        # Distinct marker for the adenoma (precursor) group (MP) --
        # NOT connected by any line to the diagnosed-cancer trend.
        for sg in PRECURSOR_STAGES:
            if sg in medians_by_stage.index:
                xi = stage_order.index(sg)
                ax.scatter([xi], [medians_by_stage[sg]], marker='*', s=180,
                           color='#b8860b', edgecolors='black', linewidth=0.6, zorder=5)

        ax.set_xticks(range(len(stage_order)))
        ax.set_xticklabels([stage_labels[s] for s in stage_order], fontsize=8, rotation=30, ha='right')
        ax.set_ylabel('log10 abundance', fontsize=9)
        ax.set_title(pa_name, fontsize=11, fontweight='bold')
        ax.set_xlabel('')

        # Per-stage n annotation, placed just above the axis floor -- guarded
        # against an all-NaN median column (old code's bare min() over
        # medians would raise ValueError in that case).
        finite_meds = [v for v in medians_by_stage.reindex(stage_order).values if pd.notna(v)]
        y_floor = (min(finite_meds) - 0.3) if finite_meds else ax.get_ylim()[0]
        for xi, sg in enumerate(stage_order):
            n = int((plot_df['Study.Group'] == sg).sum())
            ax.text(xi, y_floor, f'n={n}', ha='center', fontsize=6, color='grey')

    for j in range(n_pa, len(axes_flat)):
        axes_flat[j].set_visible(False)

    plt.suptitle(
        'Polyamine Abundance by CRC Stage Group (cross-sectional; median box + strip)\n'
        'Red diamonds = diagnosed-cancer trend (Stage 0 -> I/II -> III/IV) | '
        'Gold star = MP (adenoma; precursor, not on the cancer-severity trend line)',
        fontsize=11, fontweight='bold')
    plt.tight_layout(rect=[0, 0, 1, 0.93])

    out_traj = FIG_DIR / 'mp_polyamine_stage_boxplot_v2.png'
    fig.savefig(out_traj, dpi=150, bbox_inches='tight')
    plt.show(); plt.close()
    print(f"Saved: {out_traj}")


## 3. Classifier: MP vs Stage III/IV

A small Random Forest, cross-validated with `RepeatedStratifiedKFold`,
distinguishing `MP` samples from `Stage_III_IV` samples on species and on
metabolite features separately. Missing values in the CLR/log-transformed
feature matrices are imputed with the **feature-wise median inside a
pipeline** (fit per-fold, not on the pooled data) rather than a blanket
`fillna(0)` -- zero is not a neutral "missing" value in CLR or log space (it
sits at the geometric mean / log-pseudocount floor, not at "typical/unknown"),
so a flat `0` fill would systematically bias imputed values toward one end
of each feature's distribution. NaN counts are reported before imputation.


In [ ]:
mask = meta['Study.Group'].isin(['Stage_III_IV', 'MP'])
meta_sub = meta[mask].copy()
meta_sub['label'] = (meta_sub['Study.Group'] == 'MP').astype(int)  # MP=1, Stage_III_IV=0

pos_n, neg_n = int(meta_sub['label'].sum()), int((1 - meta_sub['label']).sum())
print(f"MP n={pos_n}, Stage_III_IV n={neg_n}")

cv = RepeatedStratifiedKFold(n_splits=5, n_repeats=5, random_state=42)

ml_rows = []
for feat_name, feat_df in [('Species', spc), ('Metabolites', mtb)]:
    sub = feat_df[feat_df['Sample'].isin(meta_sub['Sample'])].copy()
    feat_cols = [c for c in sub.columns if c != 'Sample']
    merged_sub = sub.merge(meta_sub[['Sample', 'label']], on='Sample')

    X_raw = merged_sub[feat_cols]
    n_nan = int(X_raw.isna().sum().sum())
    n_total = X_raw.size
    print(f"\n[{feat_name}] feature matrix: {X_raw.shape[0]} samples x {X_raw.shape[1]} features -- "
          f"{n_nan}/{n_total} values NaN ({n_nan / n_total:.2%}) before imputation")

    X = X_raw.values
    y = merged_sub['label'].values

    if len(np.unique(y)) < 2 or min(pos_n, neg_n) < cv.get_n_splits() // cv.n_repeats:
        print(f"[{feat_name}] Skipping -- insufficient samples per class for {cv.get_n_splits()}-fold CV.")
        continue

    pipe = Pipeline([
        ('impute', SimpleImputer(strategy='median')),  # fit per-fold inside cross_val_score, no leakage
        ('rf', RandomForestClassifier(n_estimators=200, class_weight='balanced',
                                       random_state=42, n_jobs=2)),
    ])
    aucs = cross_val_score(pipe, X, y, cv=cv, scoring='roc_auc')

    print(f"[{feat_name}] RF AUC: {aucs.mean():.3f} +/- {aucs.std():.3f}  "
          f"(n_folds={len(aucs)}, range=[{aucs.min():.3f}, {aucs.max():.3f}])")

    ml_rows.append({
        'Contrast': 'MP vs Stage_III_IV', 'Features': feat_name, 'Model': 'RF (median-imputed pipeline)',
        'N_MP': pos_n, 'N_StageIIIIV': neg_n,
        'AUC_mean': round(float(aucs.mean()), 3), 'AUC_std': round(float(aucs.std()), 3),
        'AUC_min': round(float(aucs.min()), 3), 'AUC_max': round(float(aucs.max()), 3),
        'N_NaN_before_impute': n_nan, 'Pct_NaN_before_impute': round(n_nan / n_total, 4) if n_total else np.nan,
    })

ml_summary = pd.DataFrame(ml_rows)
print("\n=== ML summary ===")
print(ml_summary.to_string(index=False) if not ml_summary.empty else "(no results)")


In [ ]:
spc_da.to_csv(TABLES_DIR / 'mp_vs_stageIIIIV_species_da_v2.csv', index=False)
mtb_da.to_csv(TABLES_DIR / 'mp_vs_stageIIIIV_metabolite_da_v2.csv', index=False)
if not pa_da.empty:
    pa_da.to_csv(TABLES_DIR / 'mp_vs_stageIIIIV_polyamine_da_v2.csv', index=False)
ml_summary.to_csv(TABLES_DIR / 'mp_trajectory_ml_auc_v2.csv', index=False)

print("Saved:")
print(f"  {TABLES_DIR / 'mp_vs_stageIIIIV_species_da_v2.csv'}")
print(f"  {TABLES_DIR / 'mp_vs_stageIIIIV_metabolite_da_v2.csv'}")
if not pa_da.empty:
    print(f"  {TABLES_DIR / 'mp_vs_stageIIIIV_polyamine_da_v2.csv'}")
print(f"  {TABLES_DIR / 'mp_trajectory_ml_auc_v2.csv'}")


## Summary

- Differential abundance (Stage III/IV vs MP) uses `MeanDiff` = mean(III/IV) − mean(MP)
  (positive = higher in Stage III/IV); the polyamine stage-group figure uses
  `utils.CRC_STAGE_ORDER` (`Healthy, MP, Stage_0, Stage_I_II, Stage_III_IV`). MP (adenoma)
  is drawn next to `Healthy`, never past `Stage_III_IV`.
- HS (history of colorectal surgery) is excluded from v2.
- All group comparisons here are cross-sectional (disjoint patients per group).
- This notebook is a self-contained leaf: it reads NB02's
  `preprocessed_data_reduced_v2.pkl` and writes only to `RESULTS_DIR/tables` and
  `RESULTS_DIR/figures/differential`. No later v2 notebook consumes these outputs.